# Exercise 1: Text Embeddings & Similarity Comparison

**Objective**: Understand how to generate embeddings using Sentence Transformers and compute similarity between sentences.

**Learning Outcomes**:
- Understand embeddings as vector representations
- Interpret similarity scores (semantic closeness)
- Use cosine similarity metric
- Visualize similarity in matrix format

## Block 1: Import Libraries

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

print("✓ All libraries imported successfully")

## Block 2: Load SentenceTransformer Model

In [ ]:
print("[Step 1] Loading SentenceTransformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ Model loaded successfully: all-MiniLM-L6-v2")
print(f"\nModel Details:")
print(f"  - Embedding Dimension: 384")
print(f"  - Model Size: Lightweight (6 layers)")
print(f"  - Best for: Fast semantic search on CPU")

## Block 3: Define Sample Input Sentences

In [ ]:
# Define sample sentences
sentences = [
    "GenAI is transforming software development",
    "Artificial Intelligence is changing how developers work",
    "I love playing cricket on weekends",
    "Machine learning is revolutionizing technology",
    "Sports and recreation are important for health"
]

print(f"[Step 2] Sample Input Sentences ({len(sentences)} sentences):")
print("-" * 80)
for i, sentence in enumerate(sentences, 1):
    print(f"{i}. {sentence}")

## Block 4: Generate Embeddings

In [ ]:
print("[Step 3] Generating embeddings...")
embeddings = model.encode(sentences, convert_to_tensor=True)

print(f"✓ Embeddings generated successfully")
print(f"\nEmbedding Shape: {embeddings.shape}")
print(f"  - Number of sentences: {embeddings.shape[0]}")
print(f"  - Dimensions per embedding: {embeddings.shape[1]}")
print(f"\nEach sentence is now represented as a {embeddings.shape[1]}-dimensional vector!")

## Block 5: Display First 5 Dimensions of Each Embedding

In [ ]:
print("=" * 80)
print("FIRST 5 DIMENSIONS OF EACH EMBEDDING")
print("=" * 80)

for i, (sentence, embedding) in enumerate(zip(sentences, embeddings), 1):
    first_5 = embedding[:5].cpu().numpy() if hasattr(embedding, 'cpu') else embedding[:5]
    print(f"\nSentence {i}: {sentence}")
    print(f"First 5 dimensions: {first_5}")
    print(f"  Values: [{', '.join([f'{val:.6f}' for val in first_5])}]")

## Block 6: Calculate Cosine Similarity Matrix

In [ ]:
# Convert embeddings to numpy for sklearn
embeddings_np = embeddings.cpu().numpy() if hasattr(embeddings, 'cpu') else np.array(embeddings)

# Calculate cosine similarity matrix
similarity_matrix = cosine_similarity(embeddings_np)

print("✓ Cosine similarity matrix calculated")
print(f"\nMatrix Shape: {similarity_matrix.shape}")
print(f"Type: {type(similarity_matrix).__name__}")

## Block 7: Display Similarity Score Matrix

In [ ]:
print("\n" + "=" * 80)
print("COSINE SIMILARITY MATRIX")
print("=" * 80)

# Create a formatted table
print(f"\n{'Sentence':30}", end="")
for j in range(len(sentences)):
    print(f"  Sent{j+1}  ", end="")
print()
print("-" * 80)

for i, sentence in enumerate(sentences):
    print(f"{sentence[:28]:30}", end="")
    for j in range(len(sentences)):
        score = similarity_matrix[i][j]
        print(f"  {score:6.4f}", end="")
    print()

print("\n" + "=" * 80)

## Block 8: Detailed Pairwise Comparison

In [ ]:
print("\n" + "=" * 80)
print("DETAILED PAIRWISE SIMILARITY COMPARISON")
print("=" * 80)

pair_data = []

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity_score = similarity_matrix[i][j]

        # Interpret the similarity
        if similarity_score > 0.7:
            interpretation = "🟢 VERY HIGH (semantically similar)"
            level = "Very High"
        elif similarity_score > 0.5:
            interpretation = "🟡 HIGH (somewhat similar)"
            level = "High"
        elif similarity_score > 0.3:
            interpretation = "🟠 MODERATE (loosely related)"
            level = "Moderate"
        else:
            interpretation = "🔴 LOW (quite different)"
            level = "Low"

        pair_data.append({
            'Pair': f"{i+1}-{j+1}",
            'Sentence 1': sentences[i],
            'Sentence 2': sentences[j],
            'Similarity Score': f"{similarity_score:.4f}",
            'Level': level
        })

        print(f"\nPair {i+1}-{j+1}:")
        print(f"  Sentence {i+1}: {sentences[i]}")
        print(f"  Sentence {j+1}: {sentences[j]}")
        print(f"  Similarity Score: {similarity_score:.4f}")
        print(f"  Interpretation: {interpretation}")

# Display as DataFrame
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)
df = pd.DataFrame(pair_data)
print(df.to_string(index=False))

## Block 9: Summary Statistics

In [ ]:
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

# Exclude diagonal elements (self-similarity)
similarity_scores_flat = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

print(f"\nTotal sentence pairs: {len(similarity_scores_flat)}")
print(f"Average similarity: {np.mean(similarity_scores_flat):.4f}")
print(f"Max similarity: {np.max(similarity_scores_flat):.4f}")
print(f"Min similarity: {np.min(similarity_scores_flat):.4f}")
print(f"Std deviation: {np.std(similarity_scores_flat):.4f}")
print(f"Median similarity: {np.median(similarity_scores_flat):.4f}")

# Find most and least similar pairs
max_idx = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)
min_idx = np.unravel_index(np.argmin(similarity_matrix), similarity_matrix.shape)

if max_idx[0] != max_idx[1]:
    print(f"\n🏆 Most similar pair: Sentence {max_idx[0]+1} & {max_idx[1]+1}")
    print(f"    Score: {similarity_matrix[max_idx]:.4f}")
    print(f"    Sent {max_idx[0]+1}: {sentences[max_idx[0]]}")
    print(f"    Sent {max_idx[1]+1}: {sentences[max_idx[1]]}")

if min_idx[0] != min_idx[1]:
    print(f"\n❄️ Least similar pair: Sentence {min_idx[0]+1} & {min_idx[1]+1}")
    print(f"    Score: {similarity_matrix[min_idx]:.4f}")
    print(f"    Sent {min_idx[0]+1}: {sentences[min_idx[0]]}")
    print(f"    Sent {min_idx[1]+1}: {sentences[min_idx[1]]}")

## Block 10: Key Insights

In [ ]:
print("\n" + "=" * 80)
print("KEY INSIGHTS & LEARNINGS")
print("=" * 80)

insights = """
1. EMBEDDINGS:
   - Each sentence is converted into a 384-dimensional vector
   - These vectors capture semantic meaning of the text
   - Similar concepts produce similar vectors

2. COSINE SIMILARITY:
   - Measures angle between vectors (not distance)
   - Range: -1 (opposite) to +1 (identical)
   - Score > 0.7: Very similar semantics
   - Score < 0.3: Quite different topics

3. SEMANTIC RELATIONSHIPS:
   - Tech-related sentences cluster together (high similarity)
   - Sports-related sentences cluster together
   - Cross-domain pairs show low similarity

4. PRACTICAL APPLICATIONS:
   - Semantic search
   - Duplicate detection
   - Document clustering
   - Recommendation systems
   - Similarity ranking

5. MODEL PERFORMANCE:
   - all-MiniLM-L6-v2 is lightweight and fast
   - Good for CPU-based deployments
   - Suitable for real-time applications
"""

print(insights)
print("=" * 80)
print("✓ Exercise 1 completed successfully!")
print("=" * 80)